In [1]:
#| default_exp bundle

In [2]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory
from setuptools.dist import Distribution
from kavacha.spec import App

What has to happen to a bundle after the freezer has run.

py2app and py2exe leave a bundle that is nearly right. What is left is the packages an archive cannot hold, the copies of those packages the archive still carries, and the icon macOS 26 draws. Each function here takes the path to a built bundle and works on the tree it finds.

Nothing here runs a freezer, and nothing here needs one installed. The examples below build the directory shape a freezer leaves and repair that.

In [3]:
#| export
from __future__ import annotations
import hashlib, os, shutil, sys, zipfile
from fastcore.all import Path, first

In [4]:
#| export
def graft(app, name, python=None):
    """Put the real `name` package into a built bundle, over whatever the freezer left of it.

    Some packages an archive simply cannot hold. `apsw` has the extension module for its own
    `__init__`, so a flattened copy loses `apsw.ext`; `playwright` carries a Node binary it has to
    *execute*, and a file inside an archive cannot be executed. Naming them in `packages` instead
    only gets `ImportError: No module named ...`, because the lookup wants an `__init__.py` to find.
    """
    import importlib.util
    spec = importlib.util.find_spec(name)
    if spec is None or not spec.submodule_search_locations:
        raise SystemExit(f'cannot graft {name}: it is not installed here')
    src = Path(first(spec.submodule_search_locations))
    lib = _lib_dir(app)
    # The package carries compiled modules built for the interpreter running this, so a bundle on
    # another version would take a wrong-ABI `.so` and fail at import with nothing to point at it.
    want = 'python%d.%d' % (python or sys.version_info[:2])
    if lib.name != want: raise SystemExit(f'cannot graft {name}: bundle is {lib.name}, this is {want}')
    dest = lib/name
    for stray in lib.glob(f'{name}.*'): stray.unlink()     # the flattened extension module
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
    return dest.relative_to(app)

def _lib_dir(app):
    "The bundle's `lib/pythonX.Y`, on either platform's layout."
    app = Path(app)
    for base in (app/'Contents'/'Resources'/'lib', app/'lib', app):
        if (hit := first(sorted(base.glob('python3.*')))) is not None: return Path(hit)
    raise SystemExit(f'{app} has no lib/python3.* to write into')

`graft` copies an installed package into a built bundle, over whatever the freezer left of it. It returns the destination relative to the bundle, which is what a build report prints.

A freezer that flattens a package whose `__init__` is an extension module leaves a lone `name.*.so` in the lib directory. That file shadows the directory that replaces it, so every `name.*` there is unlinked first. `__pycache__` directories and `.pyc` files are not copied.

`SystemExit` where the package is not installed here, and where the bundle's `lib/pythonX.Y` names a different version from the interpreter doing the copying. `python` overrides the version compared against, as a `(major, minor)` tuple.

`_lib_dir` looks for `python3.*` under `Contents/Resources/lib`, then under `lib`, then directly inside the bundle, and raises `SystemExit` where there is none.

In [5]:
#| hide
tmp = TemporaryDirectory(); root = Path(tmp.name)
ver = 'python%d.%d' % sys.version_info[:2]
zipname = 'python%d%d.zip' % sys.version_info[:2]
def mkapp(name, version=None):
    "The directory shape py2app leaves, without running py2app."
    lib = root/name/'Contents'/'Resources'/'lib'/(version or ver)
    lib.mkdir(parents=True)
    return root/name, lib

In [6]:
app, lib = mkapp('Demo.app')
(lib/'json.cpython-311-x86_64-linux-gnu.so').write_text('what the freezer flattened')
graft(app, 'json')

Path('Contents/Resources/lib/python3.11/json')

The other layouts, found by the same walk.

In [7]:
(root/'Win'/'lib'/ver).mkdir(parents=True)
_lib_dir(root/'Win').relative_to(root)

Path('Win/lib/python3.11')

In [8]:
#| hide
test_eq(list(lib.glob('json.*so')), [])
assert (lib/'json'/'decoder.py').exists(), 'the real package is in place'
test_eq(list((lib/'json').rglob('__pycache__')), [])
test_fail(lambda: graft(app, 'json', python=(3, 99)), contains='bundle is', exc=SystemExit)
test_fail(lambda: graft(app, 'no_such_package_xyz'), contains='not installed', exc=SystemExit)
test_fail(lambda: _lib_dir(root/'Empty'), contains='no lib/python3.*', exc=SystemExit)

In [9]:
#| export
def link_duplicates(app, names, floor=1_000_000):
    """Hardlink identical files across the grafted packages. Returns the megabytes saved.

    `patchright` is a fork of `playwright` and ships the same 121MB Node binary, byte for byte, so
    grafting both writes it twice. A hardlink leaves two real, executable files and one copy of the
    bytes; these are read-only library files, so nothing can write through one and surprise the
    other.
    """
    lib = _lib_dir(app)
    seen, saved = {}, 0
    for name in names:
        for f in sorted((lib/name).rglob('*')):
            if not f.is_file() or f.is_symlink() or f.stat().st_size < floor: continue
            key = (f.stat().st_size, hashlib.sha256(f.read_bytes()).hexdigest())
            if (other := seen.get(key)) is None: seen[key] = f; continue
            if os.stat(f).st_ino == os.stat(other).st_ino: continue
            saved += f.stat().st_size
            f.unlink(); os.link(other, f)
    return round(saved / 1e6)

`link_duplicates` replaces identical files across the named packages with hardlinks, and returns the megabytes recovered.

Two files are identical when their size and their SHA-256 both match. Files under `floor` bytes are never read, so hashing does not walk a whole package tree to recover a few hundred bytes. Symlinks and directories are skipped. A pair already sharing an inode is skipped, so running this twice over one bundle recovers nothing the second time.

Both paths stay real files, readable and executable. A write through one would change the other, which is safe here because these are read-only library files.

In [10]:
app2, lib2 = mkapp('Link.app')
node = os.urandom(2_000_000)                             # the Node binary both forks ship
for n in ('playwright', 'patchright'):
    (lib2/n).mkdir(); (lib2/n/'node').write_bytes(node)
link_duplicates(app2, ['playwright', 'patchright'])

2

In [11]:
#| hide
test_eq(os.stat(lib2/'playwright'/'node').st_ino, os.stat(lib2/'patchright'/'node').st_ino)
test_eq((lib2/'patchright'/'node').read_bytes(), node)   # and both are still whole files
test_eq(link_duplicates(app2, ['playwright', 'patchright']), 0)
app3, lib3 = mkapp('Diff.app')
for n, ch in (('a', b'x'), ('b', b'y')):
    (lib3/n).mkdir(); (lib3/n/'blob').write_bytes(ch * 2_000_000)
test_eq(link_duplicates(app3, ['a', 'b']), 0)            # same size, different bytes
test_eq(link_duplicates(app2, ['playwright', 'patchright'], floor=3_000_000), 0)

In [12]:
#| export
def strip_zip(app, prefixes=()):
    """Rewrite the bundle's zip without entries nothing can read. Returns the megabytes saved.

    Anything grafted is in the archive twice over, and some entries were never reachable: a package
    that finds its own data with `Path(__file__).parent/...` and `.exists()` gets False for every
    path inside an archive, so those megabytes are carried and never opened.
    """
    z = first(sorted(_lib_dir(app).parent.glob('python3*.zip')))
    if not z or not prefixes: return 0
    prefixes = tuple(prefixes)
    keep, dropped = [], 0
    with zipfile.ZipFile(z) as src:
        for info in src.infolist():
            if info.filename.startswith(prefixes): dropped += info.compress_size; continue
            keep.append((info, src.read(info.filename)))
    if not dropped: return 0
    tmp = Path(str(z) + '.new')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as out:
        for info, data in keep: out.writestr(info, data)
    tmp.replace(z)
    return round(dropped / 1e6)

`strip_zip` rewrites the bundle's `python3*.zip` without the entries whose names start with one of `prefixes`, and returns the megabytes recovered.

Those are string prefixes of the entry name, not path components, so `json` drops `jsonschema/` as well. `finish` names each grafted package as `name/` for that reason.

The saving is measured in compressed bytes, which is what the entries cost inside the archive. 0 where nothing matched, where `prefixes` is empty, and where the bundle has no archive. The archive is left byte for byte as it was in each of those.

In [13]:
app4, lib4 = mkapp('Zip.app')
z = lib4.parent/zipname
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as f:
    f.writestr('json/decoder.py', os.urandom(3_000_000))  # grafted, so in the bundle twice over
    f.writestr('jsonschema/_types.py', 'another package that starts with json')
    f.writestr('kavacha/window.py', 'the app itself')
strip_zip(app4, ('json/',))

3

In [14]:
with zipfile.ZipFile(z) as f: kept = f.namelist()
kept

['jsonschema/_types.py', 'kavacha/window.py']

In [15]:
#| hide
before = z.stat().st_mtime_ns
test_eq(strip_zip(app4, ('absent/',)), 0)
test_eq(strip_zip(app4, ()), 0)
test_eq(z.stat().st_mtime_ns, before)                    # no match, no rewrite
test_eq(strip_zip(mkapp('NoZip.app')[0], ('anything/',)), 0)   # a bundle with no archive
strip_zip(app4, ('json',))                               # without the slash, jsonschema goes too
with zipfile.ZipFile(z) as f: test_eq(f.namelist(), ['kavacha/window.py'])

In [16]:
#| export
def install_modern_icon(app, icon, identity=None, deployment_target='14.0'):
    """Compile an Icon Composer document into a built bundle, after the freezer and before signing.

    The `iconfile` in the spec is the ICNS fallback, which every freezer installs and every macOS
    before 26 reads. The Icon Composer document is what 26 draws, and its plist entries must reach
    `Info.plist` before a signature covers them. None when the document or the tooling is absent:
    a machine without Xcode 26 still builds a working app, with the older icon.
    """
    icon = Path(icon) if icon else None
    if icon is None or not icon.is_dir(): return None
    try: from iconmage import install_icon
    except ImportError:
        print('  iconmage not installed; keeping the ICNS icon alone '
              '(uv tool install git+https://github.com/AnswerDotAI/iconmage.git)')
        return None
    try: return install_icon(icon, app, deployment_target=deployment_target, identity=identity)
    except Exception as e:
        print(f'  Icon Composer icon not installed ({type(e).__name__}: {e}); keeping the ICNS one')
        return None

`install_modern_icon` compiles an Icon Composer document into a built bundle. It runs after the freezer and before signing, because the `Info.plist` entries it writes have to be inside what the signature covers. The `iconfile` in the spec is the ICNS fallback, which every freezer installs and every macOS before 26 reads.

`None` where `icon` is empty, where `icon` is not a directory, where `iconmage` is not installed, and where the compile raises. It never raises, and it never touches the ICNS the freezer installed. A machine without Xcode 26 builds a working app with the older icon.

In [17]:
(root/'Demo.icon').mkdir()
install_modern_icon(app, root/'Demo.icon') is None

  iconmage not installed; keeping the ICNS icon alone (uv tool install git+https://github.com/AnswerDotAI/iconmage.git)


True

In [18]:
#| hide
test_is(install_modern_icon(app, ''), None)
test_is(install_modern_icon(app, None), None)
test_is(install_modern_icon(app, root/'absent.icon'), None)

In [19]:
#| export
def finish(app, spec, identity=None):
    "Everything a bundle needs after the freezer wrote it, in the order it needs it."
    out = {}
    for name in spec.grafted: out[name] = str(graft(app, name))
    dropped = [*spec.unreachable, *[f'{n}/' for n in spec.grafted]]
    out['stripped_mb'] = strip_zip(app, dropped)
    out['linked_mb'] = link_duplicates(app, spec.grafted)
    out['modern_icon'] = bool(install_modern_icon(app, spec.modern_icon, identity))
    return out

`finish` runs the repairs in the order they depend on each other. Grafting comes first, because the strip drops the archive's copy of what was just grafted. Hardlinking comes after the graft, because there is nothing to link before it. The icon comes last, before signing.

The returned dict maps each grafted package to its destination, and carries the megabytes each of the two savings recovered and whether the modern icon was installed.

In [20]:
app5, lib5 = mkapp('Fin.app')
with zipfile.ZipFile(lib5.parent/zipname, 'w', zipfile.ZIP_DEFLATED) as f:
    f.writestr('json/decoder.py', os.urandom(2_000_000))
    f.writestr('test/test_json.py', os.urandom(1_000_000))
    f.writestr('kavacha/window.py', 'the app itself')
finish(app5, App('Demo', 'app.py', grafted=['json'], unreachable=['test/']))

{'json': 'Contents/Resources/lib/python3.11/json',
 'stripped_mb': 3,
 'linked_mb': 0,
 'modern_icon': False}

The graft is a real directory in `lib`, and the archive keeps neither the copy of it nor the test suite nothing imports.

In [21]:
with zipfile.ZipFile(lib5.parent/zipname) as f: left = f.namelist()
left, (lib5/'json'/'decoder.py').exists()

(['kavacha/window.py'], True)

In [22]:
#| export
def frozen_distribution():
    """A setuptools `Distribution` with no `install_requires`, which py2app 0.28.10+ refuses.

    A build run from a project root makes setuptools read `pyproject.toml` and fill the field in
    from `[project] dependencies`. A frozen build wants none of it: the spec's `packages` and
    `includes` say what the bundle carries, because the scanner cannot see an import made inside a
    function.
    """
    from setuptools.dist import Distribution
    class Frozen(Distribution):
        def parse_config_files(self, *args, **kwargs):
            super().parse_config_files(*args, **kwargs)
            self.install_requires = []
    return Frozen

`frozen_distribution` returns a `setuptools.Distribution` subclass whose `install_requires` is empty once configuration has been read. py2app 0.28.10 and later refuse a distribution that has one.

A build run from a project root makes setuptools read `pyproject.toml` and fill the field in from `[project] dependencies`. A frozen bundle installs nothing at run time. What it carries is what the spec's `packages` and `includes` name, because a scanner cannot see an import made inside a function.

In [23]:
d = frozen_distribution()({'name': 'demo', 'install_requires': ['fastcore']})
d.parse_config_files()
d.install_requires

[]

In [24]:
#| hide
kw = {'name': 'demo', 'install_requires': ['fastcore']}
plain = Distribution(kw); plain.parse_config_files()
frozen = frozen_distribution()(kw); frozen.parse_config_files()
assert plain.install_requires, 'a plain distribution keeps whatever configuration left it'
test_eq(frozen.install_requires, [])   # only the subclass empties it, wherever the build runs from

In [25]:
#| hide
tmp.cleanup()